# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [7]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Interdisciplinario"
df = get_dataset_to_split(df, feat_col)

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Interdisciplinario"])
savepath = os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)

Test size: 217
Fold 0 - Val size: 289
Archivo guardado exitosamente en /tmp/final_project/dataSplits/interdiciplinario/train_test_ids_3folds.json


# 2) 

In [10]:
import json

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/interdiciplinario/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)

In [12]:
df.columns

Index(['Código VRID', 'Línea de investigación', 'Facultad del Proyecto',
       'Depto Persona', 'Título', 'Keywords', 'Resumen', 'Titulo_trad',
       'Resumen_trad', 'keywords_trad', 'Facultad_del_Proyecto_trad',
       'Depto_Persona_trad', 'Línea_investigacion_trad', 'Español'],
      dtype='object')

In [11]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

#Lectura de codigos VRID Test
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                 test_col="Interdiciplinario")

#Lectura de codigos VRID Train
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                      test_col="Interdiciplinario")
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


KeyError: 'Interdiciplinario'